## 面试问题

循环成本与延迟模型：端到端=Σ每步，怎么压缩步数？

## 回答主线

端到端成本 = Σ(每步 模型+工具+开销)。优化优先压步数(步数是乘数)：合并独立动作、去往返、缓存。先测量(每步分解)再优化。本 Notebook 建 5 步循环的成本模型，识别可合并的两个独立只读步，合并为 4 步，对比优化前后总成本。

## 真实案例

5 步循环每步含 model/tool/overhead 成本，第 2、3 步是独立只读步可合并。合并后共享一次模型调用和开销。数据为教学成本模型，不代表真实计费。

In [1]:
steps_cost = [  # 5 步循环每步的成本分解。
    {"step": 0, "model": 5, "tool": 2, "overhead": 1},  # 第0步成本。
    {"step": 1, "model": 5, "tool": 8, "overhead": 1},  # 第1步工具较贵。
    {"step": 2, "model": 5, "tool": 2, "overhead": 1},  # 第2步只读可与第3步合并。
    {"step": 3, "model": 5, "tool": 2, "overhead": 1},  # 第3步只读可与第2步合并。
    {"step": 4, "model": 5, "tool": 3, "overhead": 1},  # 第4步。
]  # 结束成本分解。

print("步数:", len(steps_cost))  # 展示步数。
for s in steps_cost:  # 逐步打印成本。
    print("  step", s["step"], "model", s["model"], "tool", s["tool"], "overhead", s["overhead"])  # 展示每步成本分解。

步数: 5
  step 0 model 5 tool 2 overhead 1
  step 1 model 5 tool 8 overhead 1
  step 2 model 5 tool 2 overhead 1
  step 3 model 5 tool 2 overhead 1
  step 4 model 5 tool 3 overhead 1


## 基线（Baseline）

反面基线：只算端到端总成本，得到一个数。它看不出成本花在哪一步、哪里可压缩。

In [2]:
def total_cost(steps):  # 只算端到端总成本不分解。
    return sum(s["model"] + s["tool"] + s["overhead"] for s in steps)  # 各步各项求和。

baseline_total = total_cost(steps_cost)  # 优化前总成本。
print("端到端总成本:", baseline_total)  # 只有一个数不知花在哪。
print("只看总数无法定位最贵步或压缩点")  # 说明缺陷。

端到端总成本: 47
只看总数无法定位最贵步或压缩点


## 失败案例与修正

只看总数无法优化。修正一是成本模型：按类别和按步分解，定位最贵步。修正二是压缩步数：合并第 2、3 两个独立只读步，共享一次模型调用与开销。

In [3]:
def cost_breakdown(steps):  # 建立成本模型：按类别与按步分解。
    by_category = {"model": 0, "tool": 0, "overhead": 0}  # 按类别累加。
    per_step = []  # 每步总成本。
    for s in steps:  # 遍历每步。
        step_total = s["model"] + s["tool"] + s["overhead"]  # 该步总成本。
        per_step.append((s["step"], step_total))  # 记录步与其成本。
        for k in by_category:  # 累加各类别。
            by_category[k] += s[k]  # 分类累加。
    return {"by_category": by_category, "per_step": per_step}  # 返回分解。

def merge_independent_steps(steps, merge_pair):  # 合并两个独立只读步为一步。
    i, j = merge_pair  # 待合并的两步索引。
    merged = dict(steps[i])  # 以第一步为基础。
    merged["tool"] = steps[i]["tool"] + steps[j]["tool"]  # 工具成本相加。
    merged["model"] = steps[i]["model"]  # 合并后共享一次模型调用。
    merged["overhead"] = steps[i]["overhead"]  # 合并后共享一次开销。
    new_steps = [s for k, s in enumerate(steps) if k not in merge_pair]  # 去掉原两步。
    new_steps.append(merged)  # 加入合并后的步。
    return new_steps  # 返回压缩后的步序列。

In [4]:
breakdown = cost_breakdown(steps_cost)  # 计算成本分解。
optimized = merge_independent_steps(steps_cost, (2, 3))  # 合并第2、3两个独立只读步。
optimized_total = total_cost(optimized)  # 优化后总成本。
print("按类别:", breakdown["by_category"])  # 展示模型/工具/开销各占多少。
print("按步:", breakdown["per_step"])  # 展示每步成本便于定位。
print("优化前步数/成本:", len(steps_cost), "/", baseline_total)  # 展示优化前。
print("优化后步数/成本:", len(optimized), "/", optimized_total)  # 展示合并后步数与成本下降。

按类别: {'model': 25, 'tool': 17, 'overhead': 5}
按步: [(0, 8), (1, 14), (2, 8), (3, 8), (4, 9)]
优化前步数/成本: 5 / 47
优化后步数/成本: 4 / 41


## 结果解读

端到端总成本 47。成本模型显示 tool 占比高、每步成本可定位。合并第 2、3 独立只读步（共享一次 model+overhead）后步数 5→4、成本 47→41，省 6。要点：步数是乘数优先压步数、先测量再优化、只合并独立动作。

In [5]:
saved = baseline_total - optimized_total  # 计算节省的成本。
print("端到端成本 优化前:", baseline_total, "优化后:", optimized_total)  # 对比总成本。
print("节省成本:", saved, "(省去合并步的一次模型调用与开销)")  # 展示节省来源。
print("步数从", len(steps_cost), "压到", len(optimized))  # 展示步数压缩。

端到端成本 优化前: 47 优化后: 41
节省成本: 6 (省去合并步的一次模型调用与开销)
步数从 5 压到 4


In [6]:
assert baseline_total == 47  # 优化前端到端总成本。
assert optimized_total == 41  # 合并独立步后总成本下降。
assert len(optimized) == 4  # 步数从 5 压到 4。
assert saved == 6  # 节省来自合并步省下的一次模型与开销。
assert breakdown["by_category"]["tool"] == 17  # 成本模型正确统计工具总成本。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
